# ICOS Obspack zarr — interactive station/gas viewer

Pick a station trigram → height (m a.g.l.) → gas, then plot the time series.

Each zarr group is named `{TRIGRAM}{HEIGHT}` (e.g. `HTM150`); the group can
contain `co2`, `ch4`, `n2o`, and/or `co` as separate time-aligned variables.

In [1]:
import pathlib
import re
from collections import defaultdict

import xarray as xr
import zarr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output

STORE = pathlib.Path("icos-obspack.zarr")
GASES = ("co2", "ch4", "n2o", "co")
GAS_UNITS = {"co2": "ppm", "ch4": "ppb", "n2o": "ppb", "co": "ppb"}

_SID_RE = re.compile(r"^([A-Z]{3})(\d+)$")

z = zarr.open_group(str(STORE), mode="r")

# Build {trigram: {height: station_id}} index from group names
_index: dict[str, dict[int, str]] = defaultdict(dict)
for sid in sorted(z.group_keys()):
    m = _SID_RE.match(sid)
    if not m:
        continue
    _index[m.group(1)][int(m.group(2))] = sid

trigrams = sorted(_index)

def gases_in(sid: str) -> list[str]:
    """Return the gases present in a station group, in canonical order."""
    grp = z[sid]
    arrs = set(grp.array_keys())
    return [g for g in GASES if g in arrs]

print(f"Store      : {STORE}")
print(f"Trigrams   : {len(trigrams)}")
print(f"Stations   : {sum(len(h) for h in _index.values())}")

Store      : icos-obspack.zarr
Trigrams   : 15
Stations   : 43


In [2]:
# ── Widgets ───────────────────────────────────────────────────────────────────

w_trigram = widgets.Dropdown(
    options=trigrams,
    value=trigrams[0] if trigrams else None,
    description="Station:",
    layout=widgets.Layout(width="180px"),
)

w_height = widgets.Dropdown(
    options=[],
    description="Height:",
    layout=widgets.Layout(width="180px"),
)

w_gas = widgets.Dropdown(
    options=[],
    description="Gas:",
    layout=widgets.Layout(width="160px"),
)

w_btn = widgets.Button(
    description="Plot",
    button_style="primary",
    layout=widgets.Layout(width="100px"),
)

w_info = widgets.HTML(value="")
out    = widgets.Output()

# ── Cascade: trigram → height → gas ───────────────────────────────────────────

def _on_trigram_change(change):
    trig = change["new"]
    heights = sorted(_index[trig])
    w_height.options = [(f"{h} m", h) for h in heights]
    w_height.value   = heights[0] if heights else None

def _on_height_change(change):
    if w_trigram.value is None or change["new"] is None:
        w_gas.options = []
        return
    sid = _index[w_trigram.value][change["new"]]
    gases = gases_in(sid)
    w_gas.options = [(g.upper(), g) for g in gases]
    w_gas.value   = gases[0] if gases else None
    # Show site metadata
    attrs = dict(z[sid].attrs)
    site = attrs.get("site_name", "")
    cc   = attrs.get("country_code", attrs.get("site_country", ""))
    lat  = attrs.get("site_latitude", attrs.get("latitude", "?"))
    lon  = attrs.get("site_longitude", attrs.get("longitude", "?"))
    pi   = attrs.get("current_pi", "")
    bits = [f"<b>{sid}</b>", site, cc and f"({cc})", f"{lat}, {lon}"]
    if pi:
        bits.append(f"PI: {pi}")
    w_info.value = "<small>" + " \u2014 ".join(b for b in bits if b) + "</small>"

w_trigram.observe(_on_trigram_change, names="value")
w_height.observe(_on_height_change,  names="value")
_on_trigram_change({"new": w_trigram.value})
_on_height_change({"new": w_height.value})

# ── Plot ──────────────────────────────────────────────────────────────────────

def _on_plot(btn):
    if w_trigram.value is None or w_height.value is None or w_gas.value is None:
        return
    sid = _index[w_trigram.value][w_height.value]
    gas = w_gas.value

    with out:
        clear_output(wait=True)
        ds = xr.open_zarr(str(STORE), group=sid, consolidated=False)
        if gas not in ds:
            print(f"{gas!r} not found in {sid}")
            return

        da     = ds[gas]
        time   = ds[f"time_{gas}"].values
        units  = da.attrs.get("units", GAS_UNITS.get(gas, ""))
        cal    = ds.attrs.get(f"{gas}_calibration_scale", "")
        site   = ds.attrs.get("site_name", "")

        title = f"{sid} ({site}) \u2014 {gas.upper()}"
        if cal:
            title += f"  [scale: {cal}]"

        print(f"{gas.upper()} samples: {len(time):,}  range: {str(time[0])[:10]} \u2192 {str(time[-1])[:10]}")

        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(time, da.values, lw=0.5, color="tab:blue")
        ax.set_ylabel(f"{gas.upper()}\n({units})")
        ax.set_title(title)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.grid(alpha=0.3)
        fig.tight_layout()
        plt.show()

w_btn.on_click(_on_plot)

display(
    widgets.HBox([w_trigram, w_height, w_gas, w_btn]),
    w_info,
    out,
)

HTML(value='<small><b>ARN10</b> — El Arenosillo — (ES) — 37.104, -6.734 — PI: José Adame</small>')

Output()